# Class 2: Build, Break, and Observe an Inference Server

**New here?** Start with [`intro.ipynb`](intro.ipynb) — Hugging Face in-notebook inference, `/healthz`, then llama.cpp in Docker (~20 min).

**Goal:** Build a naive server, break it under load, put a gateway in front of a **real engine** (llama.cpp), and reflect on gateway vs engine responsibilities.

**Duration:** 50–70 minutes (Part 3 runs long — see timing below)

### Timing — pick a path

| Path | Parts | Skip if short on time |
|------|-------|------------------------|
| **Core (~50 min)** | 0 → 2 → 3 → 4–5 → 7 | **Part 1** (Class 1 covered TTFT), **Part 6** |
| **Full (~70 min)** | All | — |

### Narrative
1. ~~Anatomy of an LLM call~~ *(skippable)*
2. Naive FastAPI + Transformers server
3. Break it (OOM / latency explosion) — **protect this**
4. RelayServe → llama.cpp *(engine swap + gateway — two changes)*
5. Observability (`/metrics`)
6. ~~LiteLLM~~ *(skippable)*
7. Reflection — **split gateway vs engine credits**

---
**Where to run:** **Modal GPU** (Mac users) or local NVIDIA GPU — see [MODAL.md](MODAL.md). Mac CPU Docker is smoke-test only.

**Docker (GPU):** `docker compose up naive-server` · Jupyter: `docker compose --profile lab up jupyter` → `http://localhost:8888` (token: `class2`)

**Setup:** `pip install -r requirements.txt` and `export PYTHONPATH=$PWD` from `class2/`

## Part 0 — Setup & device check

**Run on Modal GPU or local NVIDIA** ([MODAL.md](MODAL.md) for Mac).

| Check | Command |
|-------|---------|
| GPU | `nvidia-smi` |
| Part 3 memory watch | `watch -n1 nvidia-smi` |
| Server (T1) | `docker compose up naive-server` |

**Skippable later:** Part 1 · Part 6

In [ ]:
import os, shutil, subprocess, sys

print("Python:", sys.version.split()[0])

if shutil.which("nvidia-smi"):
    print("nvidia-smi: OK — run the full lab here (Modal GPU or local NVIDIA)")
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
        check=False,
    )
elif sys.platform == "darwin":
    print("Mac local: no nvidia-smi — run Parts 0-7 on Modal GPU (see MODAL.md)")
else:
    print("No nvidia-smi — use Modal GPU or install NVIDIA drivers")

try:
    import torch
    print("Torch CUDA:", torch.cuda.is_available())
except ImportError:
    print("Torch not installed in this kernel")

## Part 1 — Anatomy of an LLM call · SKIPPABLE

Skip this section if Class 1 covered TTFT or you're on the 50-min core path.

Streaming: measure `ttft_ms` (first token) and `total_ms`. Non-streaming: only `total_ms` — `ttft_ms` is not measured (null).

Start the naive server first, then run the cell below.

In [ ]:
import os
os.environ.setdefault("OPENAI_BASE_URL", "http://127.0.0.1:8000/v1")

!python scripts/part1_anatomy.py

## Part 2 — Naive inference server

**T1:** server already running from Part 0 — don't restart.

**This part:** read `naive_server/server.py`, name what's missing. **Not** load testing yet.

**Optional T2 smoke test** (prints JSON only — nothing else opens):

In [ ]:
from lib.client_utils import make_client, timed_chat_completion, print_timing

client = make_client()
msgs = [{"role": "user", "content": "Say hello in five words."}]
r = timed_chat_completion(client, msgs, stream=True, max_tokens=16)
print_timing("smoke", r)

## Part 3 — Break the server · CRITICAL

**4 terminals:** T1 server (already up) · **T3 metrics first** · **T4 `nvidia-smi`** · **T2 break script last**

```text
Order:  T3 → T4 → T2   (T2 only while T3 is still printing lines)
```

**T3** — start first:

```bash
python scripts/part5_observability.py --url http://127.0.0.1:8000 --interval 1 --count 120
```

**T4** — `watch -n1 nvidia-smi` (separate terminal; not opened by curl)

**T2** — while T3 runs:

```bash
python scripts/part3_break_server.py
```

**Save:** script table + T3 screenshot (`active` > 0) + T4 screenshot during load.

In [ ]:
!python scripts/part3_break_server.py

### Why it breaks

- Each concurrent request holds **its own KV cache** during `generate()`.
- `MODEL_LOCK` serializes GPU compute but **does not limit memory** from accepted requests.
- No continuous batching → low GPU util, high latency — the worst of both worlds.
- Long context increases prefill memory **and** KV size per request.

## Part 4 — RelayServe → llama.cpp

RelayServe is the **gateway**. llama.cpp is the **engine**. Part 4 changes **both** — not just adding a gateway.

| | Arm A (Parts 2–3) | Arm B (Part 4) |
|---|---|---|
| Stack | Naive FastAPI + Transformers | RelayServe → llama.cpp ×2 |
| OOM fix? | — | Mostly **engine swap** — gateway alone would NOT fix Part 3 |

**Counterfactual:** RelayServe routing to the *same* naive Transformers process = same OOM. Don't credit the gateway with the engine's job.

**Setup:**
```bash
python scripts/download_gguf.py
docker compose --profile relay up
python scripts/part4_compare.py --concurrency 8
```

In [ ]:
import os
os.environ["OPENAI_BASE_URL"] = "http://127.0.0.1:8080/v1"

!python scripts/part4_compare.py --concurrency 8

**After the comparison — split your credits:**

| Credit the **engine** (llama.cpp) | Credit the **gateway** (RelayServe) |
|---|---|
| Fewer OOM / errors | `queue_depth`, `queue_ms` in `/metrics` |
| Different KV memory handling | Routing across :8081 / :8082 |
| Inference loop / tok/s | Observability for alerting |

Check: `curl http://127.0.0.1:8080/metrics | jq`

## Part 5 — Observability

Poll `/metrics` while load is running. Alert on TTFT p95, queue depth, OOM — **not GPU util alone**.

In [ ]:
import asyncio, json
from lib.client_utils import fetch_metrics

for url in ["http://127.0.0.1:8000", "http://127.0.0.1:8080"]:
    try:
        m = asyncio.run(fetch_metrics(url + "/v1"))
        print(url, json.dumps(m, indent=2)[:800], "...\n")
    except Exception as e:
        print(url, "unavailable:", e)

## Part 6 — LiteLLM · SKIPPABLE

Skip on the 50-min core path. Optional demo of a feature-rich gateway (auth, logging, multi-provider).

```bash
docker compose --profile litellm up
python scripts/part6_litellm_demo.py
```

| | RelayServe (Part 4) | LiteLLM (Part 6) |
|---|---|---|
| Purpose | Minimal gateway + **real engine** | Feature-rich gateway demo |
| Engine | llama.cpp | naive server (same broken engine) |
| Lesson | Correct layering | What production gateways add |

## Part 7 — Reflection

1. What broke first in the naive server and **why**?
2. **Split:** What did **llama.cpp** improve vs what did **RelayServe** improve?
3. Counterfactual: What if RelayServe routed to the naive server?
4. What metrics would you alert on in production?

In [ ]:
answers = {
    "q1_what_broke": input("1) What broke first on the naive server and why? "),
    "q2_engine_vs_gateway": input("2) What did llama.cpp improve vs what did RelayServe improve? "),
    "q3_counterfactual": input("3) What if RelayServe routed to the naive server? "),
    "q4_alerts": input("4) Production alert metrics? "),
}
print("\n--- Save these for your submission ---")
for k, v in answers.items():
    print(f"{k}: {v}")